# Emma's Librarian: eficácia dos modelos no harness (Colab)

Roda os 72 casos de `core/harness/eval/cases.jsonl` em cada modelo, com o **mesmo código, as mesmas ferramentas, o
mesmo prompt, a mesma seed e a mesma nota (v2)** da máquina local. O que muda é só o motor: `llama.cpp` com CUDA numa
GPU do Colab, que processa o prompt de ferramentas muito mais rápido.

**O que este notebook mede e o que não mede.** Mede *eficácia*: o modelo faz o certo na biblioteca? Os **tempos daqui
não valem** para decidir se o modelo é viável na RX 580: isso se mede na máquina local, só nos modelos que passarem
aqui. A última célula separa os candidatos.

Como usar:
1. No seu computador, gere o zip do código (commit atual) e coloque na pasta `emmas-eval` do seu Drive:
   `git archive --format=zip -o emmas-harness-eval.zip HEAD:emmas_librarian`
2. Em Ambiente de execução, escolha **GPU (T4)**.
3. Rode as células em ordem. Os resultados vão para o Drive e a execução é **retomável**: se a sessão cair, rode de
   novo e ela pula o que já terminou.

In [ ]:
# 1. A GPU
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout or 'SEM GPU: Ambiente de execução > Alterar tipo > GPU')

In [ ]:
# 2. CONFIGURAÇÃO (o que quiser mudar fica aqui)
ZIP_NAME = 'emmas-harness-eval.zip'
DRIVE_DIR = '/content/drive/MyDrive/emmas-eval'      # o zip fica aqui; os resultados vão para DRIVE_DIR/runs
MODELS = ['lfm3-1.7b', 'lfm2.5-8b-a1b', 'qwen3.5-2b', 'qwen3.5-4b', 'gemma-4-e4b', 'qwen3.5-9b',
          'lfm2.5-1.2b-instruct', 'lfm2-1.2b-tool', 'lfm2.5-350m']
MODES = ['routed', 'full']       # routed: o modelo escolhe os assuntos antes; full: recebe as 81 ferramentas
CASES = 'all'                    # 'all' ou ids separados por vírgula, ex.: 'art-2,proj-1'
LABEL = 'colab-eficacia'
PASS_THRESHOLD = 0.70            # taxa de acerto mínima para o modelo ir ao teste de tempo local
LLAMA_BUILD = 'b12425'

# Modelos: id -> (repositório no Hugging Face, arquivo Q4_K_M). Os ids são os de modelCatalog.ts.
CATALOG = {
    'lfm3-1.7b': ('LiquidAI/LFM3-1.7B-GGUF', 'LFM3-1.7B-Q4_K_M.gguf'),
    'lfm2.5-1.2b-instruct': ('LiquidAI/LFM2.5-1.2B-Instruct-GGUF', 'LFM2.5-1.2B-Instruct-Q4_K_M.gguf'),
    'lfm2-1.2b-tool': ('LiquidAI/LFM2-1.2B-Tool-GGUF', 'LFM2-1.2B-Tool-Q4_K_M.gguf'),
    'lfm2.5-350m': ('LiquidAI/LFM2.5-350M-GGUF', 'LFM2.5-350M-Q4_K_M.gguf'),
    'lfm2.5-8b-a1b': ('LiquidAI/LFM2.5-8B-A1B-GGUF', 'LFM2.5-8B-A1B-Q4_K_M.gguf'),
    'qwen3.5-2b': ('unsloth/Qwen3.5-2B-GGUF', 'Qwen3.5-2B-Q4_K_M.gguf'),
    'qwen3.5-4b': ('unsloth/Qwen3.5-4B-GGUF', 'Qwen3.5-4B-Q4_K_M.gguf'),
    'qwen3.5-9b': ('unsloth/Qwen3.5-9B-GGUF', 'Qwen3.5-9B-Q4_K_M.gguf'),
    'gemma-4-e4b': ('unsloth/gemma-4-E4B-it-GGUF', 'gemma-4-E4B-it-Q4_K_M.gguf'),
}

In [ ]:
# 3. Drive e código do projeto
from google.colab import drive
import os, zipfile
drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/runs', exist_ok=True)
APP = '/content/emmas'
if not os.path.exists(f'{APP}/package.json'):
    zipfile.ZipFile(f'{DRIVE_DIR}/{ZIP_NAME}').extractall(APP)
print(sorted(os.listdir(APP))[:12])

In [ ]:
# 4. Node 20 e as dependências (sem o Electron: --ignore-scripts)
!node --version 2>/dev/null | grep -q '^v2' || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1)
!node --version
%cd /content/emmas
!npm ci --ignore-scripts --no-audit --no-fund 2>&1 | tail -3
!npm rebuild better-sqlite3 2>&1 | tail -2

In [ ]:
# 5. Verificação: a biblioteca de teste F sobe em SQLite real neste sistema?
!cd /content/emmas && npx vitest run core/harness/__tests__/libraryF.test.ts 2>&1 | tail -8

In [ ]:
# 6. llama.cpp com CUDA (binário pronto, sem compilar)
BASE = f'https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}'
!mkdir -p /content/llama && cd /content/llama && \
  curl -sL -o l.tgz {BASE}/llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz && \
  curl -sL -o c.tgz {BASE}/cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz && \
  tar xzf l.tgz && tar xzf c.tgz && rm l.tgz c.tgz
import glob, os
SERVER = glob.glob('/content/llama/**/llama-server', recursive=True)[0]
LLAMA_LIB = os.path.dirname(SERVER)
os.environ['LD_LIBRARY_PATH'] = LLAMA_LIB + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.chmod(SERVER, 0o755)
print(SERVER)
!{SERVER} --list-devices 2>&1 | tail -4

In [ ]:
# 7. Modelos (direto do Hugging Face para o disco do Colab; o hash vai para o registro no Drive)
import hashlib, json, subprocess, time, datetime
os.makedirs('/content/models', exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for model in MODELS:
    repo, name = CATALOG[model]
    dest = f'/content/models/{name}'
    if os.path.exists(dest):
        continue
    started = time.time()
    subprocess.run(['curl', '-L', '--fail', '-sS', '-o', dest, f'https://huggingface.co/{repo}/resolve/main/{name}'], check=True)
    record = {'at': datetime.datetime.now().isoformat(timespec='seconds'), 'what': f'{repo}/{name}', 'bytes': os.path.getsize(dest),
              'seconds': round(time.time() - started, 1), 'sha256': sha256(dest)}
    open(f'{DRIVE_DIR}/runs/setup-colab.jsonl', 'a').write(json.dumps(record) + '\n')
    print(record['what'], record['bytes'] // 10**6, 'MB', record['seconds'], 's')

In [ ]:
# 8. A bateria: cada modelo, cada modo. Retomável: o que já está no índice do Drive é pulado.
import subprocess, json, os

INDEX = f'{DRIVE_DIR}/runs/index.jsonl'
EXPECTED = None if CASES != 'all' else 72

def already_done(model, mode):
    if not os.path.exists(INDEX):
        return False
    for line in open(INDEX):
        d = json.loads(line)
        if d.get('model') == model and d.get('mode') == mode and d.get('label') == f'{LABEL}-{mode}':
            return EXPECTED is None or d.get('cases') == EXPECTED
    return False

env = dict(os.environ, LLAMA_SERVER=SERVER, LLAMA_MODELS_DIR='/content/models', LLAMA_DEVICE='',
           LLAMA_BACKEND='cuda-12.8', LLAMA_BUILD=LLAMA_BUILD, EVAL_RUNS_DIR=f'{DRIVE_DIR}/runs')

for mode in MODES:
    for model in MODELS:
        if already_done(model, mode):
            print('já feito:', model, mode)
            continue
        print(f'=== {model} / {mode}')
        process = subprocess.Popen(['npx', 'vite-node', 'core/harness/eval/run/runEval.ts', '--', '--model', model, '--cases', CASES,
                                    '--mode', mode, '--label', f'{LABEL}-{mode}'], cwd=APP, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        log = open(f'{DRIVE_DIR}/runs/log-{model}-{mode}.txt', 'w')
        for line in process.stdout:
            log.write(line)
            if line.startswith(('PASS', 'FAIL')):
                print(line.rstrip())
        process.wait()
        log.close()
        print('saída', process.returncode)

In [ ]:
# 9. Resultado: taxa de acerto por modelo e modo, por tipo de caso, e quais vão ao teste de tempo local
import json, pandas as pd, glob
rows, kinds = [], []
for line in open(INDEX):
    d = json.loads(line)
    if not str(d.get('label', '')).startswith(LABEL):
        continue
    rows.append({'modelo': d['model'], 'modo': d['mode'], 'casos': d['cases'], 'passou': d['passed'],
                 'taxa': round(d['passed'] / d['cases'], 2), 'chamadas ao modelo': d['modelCalls'],
                 'tokens gerados': d['generatedTokens']})
    for r in d['results']:
        kinds.append({'modelo': d['model'], 'modo': d['mode'], 'tipo': r['kind'], 'passou': r['pass']})
table = pd.DataFrame(rows).sort_values(['modo', 'taxa'], ascending=[True, False])
display(table)
display(pd.DataFrame(kinds).pivot_table(index=['modelo', 'modo'], columns='tipo', values='passou', aggfunc='mean').round(2))
best = table.groupby('modelo')['taxa'].max()
candidates = sorted(best[best >= PASS_THRESHOLD].index)
print('Vão ao teste de tempo local (taxa >= %.0f%% em algum modo):' % (PASS_THRESHOLD * 100), candidates)
print('Ficam de fora:', sorted(set(best.index) - set(candidates)))
json.dump({'limiar': PASS_THRESHOLD, 'candidatos': candidates, 'taxa_maxima': best.to_dict()},
          open(f'{DRIVE_DIR}/runs/candidatos.json', 'w'), indent=2)

In [ ]:
# 10. Os casos que cada candidato errou (para ver o que o modelo faz de errado antes de decidir)
fails = {}
for line in open(INDEX):
    d = json.loads(line)
    if str(d.get('label', '')).startswith(LABEL):
        fails[(d['model'], d['mode'])] = [r['id'] for r in d['results'] if not r['pass']]
for key, ids in sorted(fails.items()):
    print(key, len(ids), ids)

In [ ]:
# 11. Baixar tudo o que foi gravado (já está no Drive; isto é só uma cópia)
import shutil
from google.colab import files
shutil.make_archive('/content/runs-colab', 'zip', f'{DRIVE_DIR}/runs')
files.download('/content/runs-colab.zip')